# 13 — Demo applicativa: squadre, mappa 2D, heatmap, distanze

Pipeline finale su una clip NBA, costruita con le scelte degli esperimenti:

**tracce (BoT-SORT oppure SAM2 + ri-detection) + interpolazione → pulizia (doppioni, piedi non visibili, cambi di squadra) → classificazione squadra → omografia → mappa 2D, heatmap, distanze**

Due sorgenti di tracce, da confrontare (`SOURCE`):
- `'botsort'`: detector migliore di E1 (RT-DETR-L ft) + BoT-SORT, miglior tracker classico di E2. Calcolato qui.
- `'sam2_redet'`: SAM2 + ri-detection, miglior metodo di E2 per identità. **Non viene ricalcolato**: legge le tracce già prodotte dal notebook 09 (`outputs/sam2_redet/<clip>/`).

Le omografie vengono lette da `outputs/geometry/<clip>/` (notebook 09), così entrambe le sorgenti usano **la stessa geometria** e non serve la chiave Roboflow; se il file manca vengono calcolate qui.

Output in `demo/<clip>/<SOURCE>/`. Esegui il notebook una volta con `SOURCE = 'botsort'` e una con `SOURCE = 'sam2_redet'`: l'ultima cella confronta le due versioni.

Nota: è una **dimostrazione qualitativa** (non ci sono ground truth per squadra e posizioni in metri). Va presentata così nella tesi.

In [ ]:
%pip install -q ultralytics trackers supervision "git+https://github.com/roboflow/sports.git@feat/basketball"
# dopo il pip: Runtime -> Riavvia sessione, poi esegui dalla cella successiva

In [ ]:
# Su Colab il pacchetto triton (usato solo per compilare modelli PyTorch, qui inutile) va in crash al momento
# dell'import e fa cadere il kernel (Segmentation fault). Lo rendiamo "non installato" per questa sessione.
import sys
if 'triton' not in sys.modules: sys.modules['triton'] = None

# ==== SETUP COMUNE (uguale in tutti i notebook) ====
import sys, os
from pathlib import Path
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/Tesi_Basket')   # progetto su Drive
    FAST = Path('/content/fast')                                 # disco locale Colab (veloce)
else:
    ROOT = Path(r'F:\Tesi Basket')                               # progetto in locale
    FAST = ROOT / '_cache'
FAST.mkdir(parents=True, exist_ok=True)
assert ROOT.is_dir(), f'Cartella del progetto non trovata: {ROOT}'
print('Progetto:', ROOT)

import json, time
import numpy as np, pandas as pd, cv2
import supervision as sv
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import faulthandler; faulthandler.enable()   # se il kernel si chiude, nel log compare la riga che lo ha causato

## 1. Configurazione

In [ ]:
SOURCE = 'sam2_redet'      # <-- 'botsort' oppure 'sam2_redet'
CLIP = ROOT / 'videos' / 'clip_05.mov'
CLIP_ID = CLIP.stem
DET_WEIGHTS = ROOT / 'models' / 'detector_best.pt'   # usato solo con 'botsort'
INTERP_MAX_GAP = 20
COURT_MODEL_ID = 'basketball-court-detection-2/14'
FT_TO_M = 0.3048

# --- pulizia delle tracce ---
DUP_IOU = 0.7              # due box della stessa scena con IoU > 0.7 = stesso giocatore: si tiene la traccia più lunga
SPLIT_ON_TEAM_CHANGE = True
SPLIT_MIN_SAMPLES = 5      # campioni consecutivi (1 ogni 3 frame ≈ 0,5 s) con il colore dell'altra squadra -> la traccia viene divisa
# zona del tabellone (frazioni di larghezza/altezza): un piede che finisce lì sotto è coperto dalla grafica
SCOREBOARD_REL = (0.00, 0.79, 0.82, 0.95)   # x1, y1, x2, y2 — controlla l'immagine 'controllo_tabellone.png'

OUT = ROOT / 'demo' / CLIP_ID / SOURCE; OUT.mkdir(parents=True, exist_ok=True)
print('Sorgente:', SOURCE, '| output:', OUT)

## 2. Video e tracce

In [ ]:
cap = cv2.VideoCapture(str(CLIP)); FPS = cap.get(cv2.CAP_PROP_FPS)
frames = []
while True:
    ok, f = cap.read()
    if not ok: break
    frames.append(f)
cap.release()
H_IMG, W_IMG = frames[0].shape[:2]
print(len(frames), 'frame', f'{W_IMG}x{H_IMG}', f'{FPS:.2f} fps')

if SOURCE == 'botsort':
    from ultralytics import YOLO, RTDETR
    import trackers as T
    model = RTDETR(str(DET_WEIGHTS)) if 'rtdetr' in str(DET_WEIGHTS).lower() else YOLO(str(DET_WEIGHTS))
    PLAYER = {i for i, n in model.names.items() if n.startswith('player')}
    tracker = T.BoTSORTTracker(frame_rate=FPS)
    rows = []
    for fi, f in enumerate(tqdm(frames, desc='detection + BoT-SORT')):
        r = model.predict(f, conf=0.05, agnostic_nms=True, verbose=False)[0]
        cls = r.boxes.cls.cpu().numpy().astype(int); xyxy = r.boxes.xyxy.cpu().numpy(); cf = r.boxes.conf.cpu().numpy()
        pl = np.isin(cls, list(PLAYER))
        det = sv.Detections(xyxy=xyxy[pl].astype(np.float32), confidence=cf[pl].astype(np.float32), class_id=np.zeros(pl.sum(), int))
        tr = tracker.update(det, frame=f)
        for b, tid in zip(tr.xyxy, tr.tracker_id):
            if tid >= 0: rows.append([fi, int(tid) + 1, *b])
    tracks = pd.DataFrame(rows, columns=['frame', 'track_id', 'x1', 'y1', 'x2', 'y2'])
elif SOURCE == 'sam2_redet':
    src = ROOT / 'outputs' / 'sam2_redet' / CLIP_ID / f'{CLIP_ID}_tracks.csv'
    assert src.is_file(), f'Tracce SAM2 non trovate: {src} (esegui prima il notebook 09)'
    t = pd.read_csv(src)
    t = t[t.mask_area_px > 0].dropna(subset=['x1', 'y1', 'x2', 'y2'])
    tracks = t[['frame', 'track_id', 'x1', 'y1', 'x2', 'y2']].astype({'frame': int, 'track_id': int}).reset_index(drop=True)
else:
    raise ValueError(SOURCE)

def interpolate(t, max_gap):
    out = []
    for tid, g in t.groupby('track_id'):
        g = g.set_index('frame').reindex(range(g.frame.min(), g.frame.max() + 1))
        miss = g.x1.isna(); run = (miss != miss.shift()).cumsum()
        g = g.interpolate(limit_area='inside')[~miss | (miss.groupby(run).transform('sum') <= max_gap)]
        g['track_id'] = tid; out.append(g.reset_index().rename(columns={'index': 'frame'}))
    return pd.concat(out, ignore_index=True)
tracks = interpolate(tracks, INTERP_MAX_GAP)
tracks['frame'] = tracks.frame.astype(int); tracks['track_id'] = tracks.track_id.astype(int)
print('tracce:', tracks.track_id.nunique(), '| righe:', len(tracks))

## 3. Pulizia: doppioni e piedi non visibili

- **Doppioni**: nello stesso frame, due box con IoU > `DUP_IOU` sono lo stesso giocatore seguito da due tracce; si elimina la riga della traccia più corta (solo in quei frame).
- **Piedi non visibili**: se il fondo della box tocca il bordo inferiore dell'immagine o finisce sul tabellone, il punto dei piedi non è affidabile; la box resta nel video ma il punto **non** viene proiettato sulla mappa.

In [ ]:
def iou_xyxy(a, b):
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    ar = lambda r: (r[:, 2] - r[:, 0]) * (r[:, 3] - r[:, 1])
    return inter / (ar(a)[:, None] + ar(b)[None, :] - inter + 1e-9)

length = tracks.groupby('track_id').size()
drop, dup_pairs = [], {}
for fi, g in tracks.groupby('frame'):
    if len(g) < 2: continue
    M = iou_xyxy(g[['x1', 'y1', 'x2', 'y2']].values, g[['x1', 'y1', 'x2', 'y2']].values); np.fill_diagonal(M, 0)
    ids, idx = g.track_id.values, g.index.values
    for i, j in zip(*np.where(np.triu(M) > DUP_IOU)):
        short = i if length[ids[i]] < length[ids[j]] else j
        drop.append(idx[short]); key = tuple(sorted((int(ids[i]), int(ids[j])))); dup_pairs[key] = dup_pairs.get(key, 0) + 1
tracks = tracks.drop(index=sorted(set(drop))).reset_index(drop=True)
print(f'Doppioni rimossi: {len(set(drop))} righe; coppie di tracce coinvolte (frame):', dup_pairs)

tracks['foot_x'] = (tracks.x1 + tracks.x2) / 2; tracks['foot_y'] = tracks.y2
sx1, sy1, sx2, sy2 = SCOREBOARD_REL[0] * W_IMG, SCOREBOARD_REL[1] * H_IMG, SCOREBOARD_REL[2] * W_IMG, SCOREBOARD_REL[3] * H_IMG
on_edge = tracks.y2 >= H_IMG - 3
on_board = tracks.foot_x.between(sx1, sx2) & tracks.y2.between(sy1 - 6, sy2)
tracks['foot_ok'] = ~(on_edge | on_board)
print(f'Punti piedi non affidabili: {(~tracks.foot_ok).sum()} su {len(tracks)} '
      f'(bordo immagine {on_edge.sum()}, tabellone {on_board.sum()})')

# immagine di controllo della zona tabellone
fi = len(frames) // 2; img = frames[fi].copy()
cv2.rectangle(img, (int(sx1), int(sy1)), (int(sx2), int(sy2)), (0, 0, 255), 2)
cv2.imwrite(str(OUT / 'controllo_tabellone.png'), img)
plt.figure(figsize=(8, 4.5)); plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.axis('off')
plt.title('Zona tabellone (rosso): se non copre la grafica, correggi SCOREBOARD_REL'); plt.show()

## 4. Classificazione della squadra (per traccia) e divisione delle tracce che cambiano squadra

Maglia descritta con un istogramma di colore Lab sul busto, dopo aver tolto i pixel del parquet; K-Means (k=2) sulla **mediana per traccia**. Poi ogni campione viene confrontato con i due centri: se una traccia mostra per almeno `SPLIT_MIN_SAMPLES` campioni consecutivi il colore dell'altra squadra, il tracker ha scambiato due avversari → la traccia viene **divisa** in quel punto (nuovo ID) invece di colorare male il giocatore.

In [ ]:
smp = np.concatenate([cv2.cvtColor(np.ascontiguousarray(f[int(.6*H_IMG):int(.78*H_IMG), int(.2*W_IMG):int(.8*W_IMG)]), cv2.COLOR_BGR2LAB).reshape(-1, 3)
                      for f in frames[::30]]).astype(np.float32)
COURT_LAB = np.median(smp, axis=0)

def jersey_feature(img, b):
    x1, y1, x2, y2 = map(int, b)
    x1, x2 = max(0, x1), min(W_IMG, x2); y1, y2 = max(0, y1), min(H_IMG, y2)   # box dentro l'immagine
    w, h = x2 - x1, y2 - y1
    if h < 30 or w < 8: return None
    crop = np.ascontiguousarray(img[y1 + int(.15*h):y1 + int(.5*h), x1 + int(.2*w):x2 - int(.2*w)])
    if crop.size == 0 or crop.ndim != 3: return None
    lab = cv2.cvtColor(crop, cv2.COLOR_BGR2LAB).reshape(-1, 3).astype(np.float32)
    lab = lab[np.linalg.norm(lab - COURT_LAB, axis=1) > 25]
    if len(lab) < 20: return None
    hist = np.histogramdd(lab, bins=(4, 6, 6), range=((0, 256), (0, 256), (0, 256)))[0].ravel()
    return hist / hist.sum()

def sample_features(t):
    F, K, R = [], [], []
    for r in t[t.frame % 3 == 0].itertuples():
        f = jersey_feature(frames[int(r.frame)], (r.x1, r.y1, r.x2, r.y2))
        if f is not None: F.append(f); K.append(r.track_id); R.append(r.frame)
    return np.array(F), np.array(K), np.array(R)

class KMeans2:
    # K-Means a 2 centri in puro numpy (evita scikit-learn, che su Colab può far cadere il kernel dopo i pip install)
    def __init__(self, n_init=20, iters=100, seed=0): self.n_init, self.iters, self.rng = n_init, iters, np.random.default_rng(seed)
    def fit(self, X):
        X = np.asarray(X, np.float64); best = None
        for _ in range(self.n_init):
            C = X[self.rng.choice(len(X), 2, replace=len(X) < 2)].copy()
            for _ in range(self.iters):
                lab = self._assign(X, C)
                newC = np.array([X[lab == k].mean(0) if (lab == k).any() else C[k] for k in range(2)])
                if np.allclose(newC, C): break
                C = newC
            inertia = ((X - C[self._assign(X, C)]) ** 2).sum()
            if best is None or inertia < best[0]: best = (inertia, C)
        self.cluster_centers_ = best[1]; return self
    @staticmethod
    def _assign(X, C): return np.argmin(((X[:, None, :] - C[None]) ** 2).sum(-1), axis=1)
    def predict(self, X): return self._assign(np.asarray(X, np.float64), self.cluster_centers_)

def fit_teams(F, K):
    med = {t: np.median(F[K == t], axis=0) for t in np.unique(K)}
    n = pd.Series(K).value_counts()
    long_ids = [t for t in med if n[t] >= 5] or list(med)
    km = KMeans2().fit(np.array([med[t] for t in long_ids]))
    return km, {int(t): int(km.predict(med[t][None])[0]) for t in med}

feats, keys, fr = sample_features(tracks)
km, team_of = fit_teams(feats, keys)

n_split = 0
if SPLIT_ON_TEAM_CHANGE:
    lab_s = km.predict(feats); next_id = int(tracks.track_id.max()) + 1
    for tid in np.unique(keys):
        sel = np.argsort(fr[keys == tid]); f_t = fr[keys == tid][sel]; l_t = lab_s[keys == tid][sel]
        other = l_t != team_of[int(tid)]
        # inizio di ogni serie di campioni consecutivi "dell'altra squadra" lunga almeno SPLIT_MIN_SAMPLES
        run_start, cnt, cuts = None, 0, []
        for k, o in enumerate(other):
            if o: run_start = k if cnt == 0 else run_start; cnt += 1
            else:
                if cnt >= SPLIT_MIN_SAMPLES: cuts.append((f_t[run_start], f_t[k]))
                cnt = 0
        if cnt >= SPLIT_MIN_SAMPLES: cuts.append((f_t[run_start], tracks.frame.max() + 1))
        for a, b in cuts:                                   # il tratto [a, b) diventa una nuova traccia
            m = (tracks.track_id == tid) & (tracks.frame >= a) & (tracks.frame < b)
            tracks.loc[m, 'track_id'] = next_id; next_id += 1; n_split += 1
    if n_split:
        feats, keys, fr = sample_features(tracks)
        km, team_of = fit_teams(feats, keys)
print('Tracce divise per cambio di squadra:', n_split)

lab_s = km.predict(feats)
purity = pd.Series({t: (lab_s[keys == t] == team_of[int(t)]).mean() for t in np.unique(keys)})
tracks['team'] = tracks.track_id.map(team_of).fillna(-1).astype(int)
long_mask = tracks.groupby('track_id').frame.transform('size') >= 2 * FPS
print('Tracce per squadra:', pd.Series(team_of).value_counts().to_dict(),
      '| tracce lunghe (>= 2 s) per squadra:', tracks[long_mask].groupby('team').track_id.nunique().to_dict())
print('Coerenza media per traccia:', round(purity.mean(), 3))
TEAM_COLORS = {0: (255, 140, 30), 1: (40, 40, 230), -1: (180, 180, 180)}   # BGR

fig, axes = plt.subplots(2, 1, figsize=(16, 5))
for ax, team in zip(axes, [0, 1]):
    crops = []
    for tid in sorted(t for t, k in team_of.items() if k == team):
        g = tracks[tracks.track_id == tid]
        if not len(g): continue
        r = g.iloc[len(g) // 2]
        c = np.ascontiguousarray(frames[int(r.frame)][max(0, int(r.y1)):min(H_IMG, int(r.y2)), max(0, int(r.x1)):min(W_IMG, int(r.x2))])
        if c.size and c.shape[0] > 1 and c.shape[1] > 1:
            c = cv2.resize(c, (60, 120)); cv2.putText(c, str(tid), (2, 14), cv2.FONT_HERSHEY_SIMPLEX, .5, (0, 255, 255), 2)
            crops.append(c)
    if crops: ax.imshow(cv2.cvtColor(np.hstack(crops), cv2.COLOR_BGR2RGB))
    ax.set_title(f'Squadra {team}'); ax.axis('off')
fig.tight_layout(); fig.savefig(OUT / 'controllo_squadre.png', dpi=150); plt.show()

## 5. Omografia immagine → campo (stessa geometria del notebook 09, fallback all'ultima valida)

In [ ]:
from sports import MeasurementUnit
from sports.basketball import CourtConfiguration, League, draw_court
court = CourtConfiguration(league=League.NBA, measurement_unit=MeasurementUnit.FEET)
V = np.asarray(court.vertices, dtype=np.float32)

HFILE = ROOT / 'outputs' / 'geometry' / CLIP_ID / f'{CLIP_ID}_homographies_image_to_court_ft.json'
if HFILE.is_file():
    hs = {int(k): np.asarray(v) for k, v in json.loads(HFILE.read_text()).items()}
    print('Omografie lette da', HFILE.name)
else:
    try:
        from inference import get_model
    except ImportError:
        raise RuntimeError('Omografie non trovate in outputs/geometry: esegui prima il notebook 09, '
                           'oppure installa inference (%pip install inference) e riavvia la sessione.')
    from getpass import getpass
    court_model = get_model(model_id=COURT_MODEL_ID, api_key=os.environ.get('ROBOFLOW_API_KEY') or getpass('API key Roboflow: '))
    hs = {}
    for fi, f in enumerate(tqdm(frames, desc='omografia')):
        res = court_model.infer(f, confidence=0.3)[0]
        if not res.predictions: continue
        pr = max(res.predictions, key=lambda p: p.confidence)
        pts = [p for p in pr.keypoints if p.confidence >= 0.5 and 0 <= p.class_id < len(V)]
        if len(pts) >= 6:
            fwd, mask = cv2.findHomography(V[[p.class_id for p in pts]], np.float32([[p.x, p.y] for p in pts]), cv2.RANSAC, 8.0)
            if fwd is not None and mask is not None and mask.sum() >= 6:
                hs[fi] = np.linalg.inv(fwd)
Hs, status, last = [], [], None
for fi in range(len(frames)):
    H = hs.get(fi)
    if H is not None: last = H; status.append('ok')
    elif last is not None: status.append('fallback')
    else: status.append('none')
    Hs.append(H if H is not None else last)
print(pd.Series(status).value_counts().to_dict())

xy = []
for r in tracks.itertuples():
    H = Hs[int(r.frame)]
    if H is None or not r.foot_ok: xy.append((np.nan, np.nan)); continue
    xy.append(tuple(cv2.perspectiveTransform(np.float32([[[r.foot_x, r.foot_y]]]), H)[0, 0]))
tracks[['x_ft', 'y_ft']] = np.array(xy)
inside = tracks.x_ft.between(0, court.court_length) & tracks.y_ft.between(0, court.court_width)
tracks.loc[~inside, ['x_ft', 'y_ft']] = np.nan
W_SM = max(3, int(round(FPS / 2)) | 1)
tracks = tracks.sort_values(['track_id', 'frame']).reset_index(drop=True)
tracks[['xs_ft', 'ys_ft']] = tracks.groupby('track_id')[['x_ft', 'y_ft']].transform(
    lambda s: s.rolling(W_SM, center=True, min_periods=1).median().rolling(5, center=True, min_periods=1).mean())
tracks.loc[tracks.x_ft.isna(), ['xs_ft', 'ys_ft']] = np.nan        # nessun punto inventato dove i piedi non sono visibili
tracks.to_csv(OUT / f'{CLIP_ID}_tracks_court.csv', index=False)

## 6. Video: clip annotata + mappa 2D affiancate

In [ ]:
SCALE, PAD = 10, 30
base = draw_court(config=court, scale=SCALE, padding=PAD)
def to_map(x, y): return int(x * SCALE) + PAD, int(y * SCALE) + PAD

mh = H_IMG; mw = int(base.shape[1] * mh / base.shape[0])
out_w = W_IMG + mw; out_w += out_w % 2
vw = cv2.VideoWriter(str(OUT / 'demo_raw.mp4'), cv2.VideoWriter_fourcc(*'mp4v'), FPS, (out_w, H_IMG))
by_frame = dict(tuple(tracks.groupby('frame')))
for fi, f in enumerate(tqdm(frames, desc='render')):
    img = f.copy(); mp = base.copy()
    for r in by_frame.get(fi, pd.DataFrame()).itertuples():
        c = TEAM_COLORS[r.team]
        cv2.rectangle(img, (int(r.x1), int(r.y1)), (int(r.x2), int(r.y2)), c, 2 if r.foot_ok else 1)
        cv2.putText(img, str(r.track_id) + ('' if r.foot_ok else '*'), (int(r.x1), max(12, int(r.y1) - 4)),
                    cv2.FONT_HERSHEY_SIMPLEX, .5, c, 2)
        if not np.isnan(r.xs_ft):
            cv2.circle(mp, to_map(r.xs_ft, r.ys_ft), 9, c, -1); cv2.circle(mp, to_map(r.xs_ft, r.ys_ft), 9, (0, 0, 0), 1)
            cv2.putText(mp, str(r.track_id), to_map(r.xs_ft - .8, r.ys_ft + .6), cv2.FONT_HERSHEY_SIMPLEX, .35, (255, 255, 255), 1)
    cv2.putText(img, f'{SOURCE}  frame {fi}', (10, 20), cv2.FONT_HERSHEY_SIMPLEX, .55, (255, 255, 255), 2)
    if status[fi] != 'ok':
        cv2.putText(mp, f'geometria: {status[fi]}', (10, 20), cv2.FONT_HERSHEY_SIMPLEX, .5, (0, 0, 255), 1)
    canvas = np.zeros((H_IMG, out_w, 3), np.uint8)
    canvas[:, :W_IMG] = img; canvas[:, W_IMG:W_IMG + mw] = cv2.resize(mp, (mw, mh))
    vw.write(canvas)
vw.release()
import subprocess
subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(OUT / 'demo_raw.mp4'), '-c:v', 'libx264', '-crf', '23',
                '-pix_fmt', 'yuv420p', str(OUT / 'demo.mp4')], check=True)
(OUT / 'demo_raw.mp4').unlink()
print('Video:', OUT / 'demo.mp4', '  (* = piedi non visibili, punto non proiettato)')

## 7. Heatmap per squadra, distanze e velocità

- **Heatmap**: istogramma 2D con celle di 1 piede, sfocato con filtro gaussiano, normalizzato per squadra.
- **Distanze/velocità**: solo tracce visibili per almeno 2 s; posizioni lisciate e campionate ogni ~0,2 s; velocità massima su finestre di 0,5 s; spostamenti > 10 m/s scartati.

In [ ]:
from scipy.ndimage import gaussian_filter
L, Wc = int(np.ceil(court.court_length)), int(np.ceil(court.court_width))
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, team in zip(axes, [0, 1]):
    d = tracks[(tracks.team == team)].dropna(subset=['xs_ft'])
    ax.imshow(cv2.cvtColor(base, cv2.COLOR_BGR2RGB))
    if len(d):
        hst, _, _ = np.histogram2d(d.ys_ft, d.xs_ft, bins=(Wc, L), range=((0, Wc), (0, L)))
        hst = gaussian_filter(hst, sigma=2.0); hst /= hst.max()
        ax.imshow(np.ma.masked_less(hst, .05), cmap='inferno', alpha=.7, vmin=0, vmax=1,
                  extent=(PAD, PAD + L * SCALE, PAD + Wc * SCALE, PAD), interpolation='bilinear')
    ax.set_xlim(0, base.shape[1]); ax.set_ylim(base.shape[0], 0)
    ax.set_title(f'{SOURCE} — squadra {team}: {d.track_id.nunique()} tracce, {len(d)/FPS:.0f} s di presenza'); ax.axis('off')
fig.tight_layout(); fig.savefig(OUT / 'heatmap_squadre.png', dpi=200); plt.show()

STEP = max(1, int(round(FPS / 5))); WIN = max(1, int(round(FPS / 2)))
stats = []
for tid, g in tracks.dropna(subset=['xs_ft']).sort_values('frame').groupby('track_id'):
    if len(g) < 2 * FPS: continue
    s_ = g.iloc[::STEP]
    dt = s_.frame.diff() / FPS
    step = np.hypot(s_.xs_ft.diff(), s_.ys_ft.diff()) * FT_TO_M
    ok = (dt > 0) & (dt <= 1) & (step / dt < 10)
    gw = g.iloc[::WIN]; vw_ = np.hypot(gw.xs_ft.diff(), gw.ys_ft.diff()) * FT_TO_M / (gw.frame.diff() / FPS)
    vw_ = vw_[vw_ < 10]
    dur = (g.frame.max() - g.frame.min()) / FPS
    stats.append({'track_id': tid, 'team': int(g.team.iloc[0]), 'secondi_visibile': round(len(g) / FPS, 2),
                  'distanza_m': step[ok].sum(), 'vel_media_m_s': step[ok].sum() / max(dur, 1e-9),
                  'vel_max_0.5s_m_s': vw_.max() if len(vw_) else np.nan,
                  'scarti_vel_implausibile': int((~ok & dt.notna()).sum())})
stats = pd.DataFrame(stats).sort_values('distanza_m', ascending=False).round(2)
stats.to_csv(OUT / 'distanze_velocita.csv', index=False)

summary = {'source': SOURCE, 'tracce': int(tracks.track_id.nunique()),
           'tracce_lunghe_2s': int(tracks[tracks.groupby('track_id').frame.transform('size') >= 2 * FPS].track_id.nunique()),
           'righe_doppioni_rimosse': len(set(drop)), 'tracce_divise_cambio_squadra': n_split,
           'punti_piedi_non_affidabili': int((~tracks.foot_ok).sum()), 'coerenza_squadra': round(float(purity.mean()), 3),
           'tracce_per_squadra': {str(k): int(v) for k, v in pd.Series(team_of).value_counts().items()},
           'vel_max_mediana_m_s': float(stats['vel_max_0.5s_m_s'].median()) if len(stats) else None}
(OUT / 'riepilogo.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
stats

## 8. Confronto BoT-SORT vs SAM2 + ri-detection (dopo aver eseguito entrambe le sorgenti)

In [ ]:
rows = []
for s in ['botsort', 'sam2_redet']:
    p = ROOT / 'demo' / CLIP_ID / s / 'riepilogo.json'
    if p.is_file(): rows.append(json.loads(p.read_text()))
if len(rows) == 2:
    display(pd.DataFrame(rows).set_index('source').T)
    fig, axes = plt.subplots(2, 1, figsize=(14, 9))
    for ax, s in zip(axes, ['botsort', 'sam2_redet']):
        ax.imshow(plt.imread(str(ROOT / 'demo' / CLIP_ID / s / 'heatmap_squadre.png'))); ax.axis('off')
    fig.tight_layout(); fig.savefig(ROOT / 'demo' / CLIP_ID / 'confronto_heatmap.png', dpi=150); plt.show()
else:
    print('Esegui il notebook anche con l\'altra SOURCE per vedere il confronto. Presenti:', [r['source'] for r in rows])